# NeMo-RL GRPO — video→DAG (25 GT) + W&B

Trains with **GRPO** on `data/generated/nemo_rl/gt.jsonl` using `dag_reward` (schema / critical ops / taken branch). Logs `train/loss`, `train/reward_*`, KL to W&B; this notebook polls live curves.

**Run on Brev** (`teachonce-gpu`). Local Mac only launches + plots.

| | |
|--|--|
| GT | 25 rows in `gt.jsonl` |
| Reward | `scripts.nemo_rl_gt.dag_reward` |
| Loop | rollout G=4 → LOO advantage → LoRA update |
| Default model | `Qwen/Qwen2.5-1.5B-Instruct` (cu124 / Brev driver 565) |

> Full NeMo-RL + Nemotron-12B-VL video rollouts need CUDA ≥12.8. This uses the same RL loop on a driver-compatible stack with **filmed events as the video proxy**. Override `GRPO_MODEL` when you have VRAM + a newer driver.

In [7]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "smoke-test":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

GT = ROOT / "data/generated/nemo_rl/gt.jsonl"
INSTANCE = os.environ.get("BREV_INSTANCE_NAME", "teachonce-gpu")
WANDB_KEY = os.environ.get("WANDB_API_KEY", "").strip()
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "teachonce-nemo-rl")
WANDB_NAME = os.environ.get("WANDB_NAME", "grpo-dag-25")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_BASE = os.environ.get("WANDB_BASE_URL", "https://api.forge.coreweave.com")
GRPO_MODEL = os.environ.get("GRPO_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
GRPO_STEPS = int(os.environ.get("GRPO_STEPS", "50"))

assert WANDB_KEY, "Add WANDB_API_KEY to .env"
assert GT.exists(), f"missing {GT} — run process_stage_b_nemo_rl.py first"
print("ROOT=", ROOT)
print("GT=", GT, "lines=", sum(1 for _ in GT.open() if _.strip()))
print("MODEL=", GRPO_MODEL, "STEPS=", GRPO_STEPS)
print("W&B=", WANDB_ENTITY or "-", WANDB_PROJECT, WANDB_NAME)

ROOT= /Users/kedarvaidya/Downloads/NvidiaHack
GT= /Users/kedarvaidya/Downloads/NvidiaHack/data/generated/nemo_rl/gt.jsonl lines= 25
MODEL= Qwen/Qwen2.5-1.5B-Instruct STEPS= 50
W&B= kkedarvaidya-duke-university teachonce-nemo-rl grpo-smoke-10


## 1. Validate GT + reward self-check

Gold DAG against itself must score ≈1.0.

In [8]:
from scripts.nemo_rl_gt import dag_reward

rows = [json.loads(l) for l in GT.read_text().splitlines() if l.strip()]
assert len(rows) == 25, f"expected 25 GT, got {len(rows)}"

bad = []
for r in rows:
    sc = dag_reward(r["gold_dag"], r)
    if sc["reward"] < 0.99:
        bad.append((r["id"], sc))

families = {}
for r in rows:
    families[r.get("family", "?")] = families.get(r.get("family", "?"), 0) + 1

print("count=", len(rows))
print("families=", families)
print("self_score_fail=", bad or None)
assert not bad, bad
print("OK — reward contract ready for GRPO")

count= 25
families= {'severity_escalation': 1, 'parts_fulfillment': 2, 'warranty_resolution': 22}
self_score_fail= None
OK — reward contract ready for GRPO


## 2. Start GRPO on Brev (no copy)

Uses GT + scripts already on the box (`~/teachonce/...`). Only exports env and `nohup`s the trainer.

```python
os.environ["GRPO_STEPS"] = "50"
os.environ["WANDB_NAME"] = "grpo-dag-25"
```

In [9]:
if not shutil.which("brev"):
    raise SystemExit("brev CLI not found")

REMOTE = "/home/ubuntu/teachonce"  # absolute — no ~, no brev copy


def brev_bash(cmd: str) -> str:
    return subprocess.check_output(["brev", "exec", INSTANCE, cmd], text=True)


# verify remote assets (already on box)
check = brev_bash(
    f"test -f {REMOTE}/data/generated/nemo_rl/gt.jsonl "
    f"&& test -f {REMOTE}/scripts/run_grpo_dag_wandb.sh "
    f"&& test -f {REMOTE}/scripts/grpo_dag_wandb.py "
    f"&& wc -l {REMOTE}/data/generated/nemo_rl/gt.jsonl "
    f"&& echo OK_REMOTE"
)
print(check)
assert "OK_REMOTE" in check, "missing GT/trainer on Brev — regenerate there, don't copy from Mac"

# free GPU from idle vLLM if present (leave stage-b CPU workers alone)
print(brev_bash("pkill -f 'vllm serve' 2>/dev/null || true; pkill -f '[p]ython -u .*grpo_dag_wandb' 2>/dev/null || true; echo GPU_CLEARED"))

env_exports = (
    f"export PATH=$HOME/.local/bin:$PATH; "
    f"export WANDB_API_KEY='{WANDB_KEY}'; "
    f"export WANDB_BASE_URL='{WANDB_BASE}'; "
    f"export WANDB_PROJECT='{WANDB_PROJECT}'; "
    f"export WANDB_NAME='{WANDB_NAME}'; "
    f"export WANDB_ENTITY='{WANDB_ENTITY}'; "
    f"export GRPO_MODEL='{GRPO_MODEL}'; "
    f"export GRPO_STEPS='{GRPO_STEPS}'; "
    f"export GRPO_GROUP='{os.environ.get('GRPO_GROUP', '4')}'; "
    f"export GRPO_MAX_NEW='{os.environ.get('GRPO_MAX_NEW', '1024')}'; "
    f"export GRPO_LORA='{os.environ.get('GRPO_LORA', '1')}'; "
    f"export GRPO_GT={REMOTE}/data/generated/nemo_rl/gt.jsonl; "
    f"export ROOT={REMOTE}; "
    f"export HF_HOME=/data/hf-cache; "
)
remote = (
    env_exports
    + f"chmod +x {REMOTE}/scripts/run_grpo_dag_wandb.sh; "
    + f"nohup bash {REMOTE}/scripts/run_grpo_dag_wandb.sh "
    + f">{REMOTE}/logs/grpo_dag_launch.out 2>&1 & echo STARTED:$!"
)
out = brev_bash(remote)
print(out)
print("project=", WANDB_PROJECT, "name=", WANDB_NAME, "steps=", GRPO_STEPS)
time.sleep(8)
print(brev_bash(f"tail -n 50 {REMOTE}/logs/grpo_dag_launch.out || true"))
print(brev_bash("pgrep -af grpo_dag || echo NO_PROC"))

KeyboardInterrupt: 

## 3. Live W&B training curves

Polls until `GRPO_STEPS` logged or the run finishes. Re-run this cell anytime.

In [ ]:
try:
    import wandb
    from tqdm.auto import tqdm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb", "matplotlib", "tqdm"])
    import wandb
    from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

os.environ["WANDB_API_KEY"] = WANDB_KEY
os.environ.setdefault("WANDB_BASE_URL", WANDB_BASE)
api = wandb.Api(timeout=30, overrides={"base_url": WANDB_BASE})

RUN_PATH = os.environ.get("WANDB_RUN_PATH", "").strip()


def find_run():
    if RUN_PATH:
        try:
            return api.run(RUN_PATH)
        except Exception:
            pass
    path = f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT
    for r in api.runs(path, order="-created_at", per_page=15):
        if r.name == WANDB_NAME or r.name.startswith(WANDB_NAME):
            return r
    return next(iter(api.runs(path, order="-created_at", per_page=1)), None)


run = None
for _ in tqdm(range(90), desc="wait W&B run", leave=False):
    try:
        run = find_run()
        if run is not None:
            break
    except Exception as e:
        tqdm.write(f"waiting for run… {e}")
    time.sleep(5)
if run is None:
    raise SystemExit(
        "No W&B run yet — check: brev exec teachonce-gpu \"tail -80 ~/teachonce/logs/grpo_dag_launch.out\""
    )

print("tracking", getattr(run, "url", run.path))
print("set WANDB_RUN_PATH=", run.path)
target_steps = GRPO_STEPS
pbar = tqdm(total=target_steps, desc="GRPO steps", unit="step")
last_n = 0

while True:
    run = api.run(run.path)
    hist = run.history(samples=2000, pandas=False)
    steps = [h.get("_step", i) for i, h in enumerate(hist)]
    loss = [h.get("train/loss") for h in hist]
    rew = [h.get("train/reward_mean") for h in hist]
    schema = [h.get("train/schema_ok") for h in hist]
    crit = [h.get("train/critical_ops") for h in hist]
    branch = [h.get("train/taken_branch") for h in hist]
    kl = [h.get("train/kl") for h in hist]

    max_step = max((s for s in steps if s is not None), default=-1)
    done_n = max_step + 1 if max_step >= 0 else 0
    if done_n > last_n:
        pbar.update(done_n - last_n)
        last_n = done_n
    if rew:
        latest_r = next((x for x in reversed(rew) if x is not None), None)
        latest_loss = next((x for x in reversed(loss) if x is not None), None)
        pbar.set_postfix(
            state=run.state,
            R=f"{latest_r:.3f}" if latest_r is not None else "-",
            loss=f"{latest_loss:.3f}" if latest_loss is not None else "-",
        )

    clear_output(wait=True)
    display(pbar.container if hasattr(pbar, "container") else pbar)
    print(getattr(run, "url", run.path))
    print(f"state={run.state} points={len(hist)} step={done_n}/{target_steps}")
    fig, ax = plt.subplots(2, 3, figsize=(13, 6))
    ax = ax.ravel()
    ax[0].plot(steps, loss, marker=".")
    ax[0].set_title("loss")
    ax[1].plot(steps, rew, marker=".", color="C2")
    ax[1].set_title("reward_mean")
    ax[2].plot(steps, kl, marker=".", color="C3")
    ax[2].set_title("kl")
    ax[3].plot(steps, schema, marker=".", color="C4")
    ax[3].set_title("schema_ok")
    ax[4].plot(steps, crit, marker=".", color="C5")
    ax[4].set_title("critical_ops")
    ax[5].plot(steps, branch, marker=".", color="C6")
    ax[5].set_title("taken_branch")
    for a in ax:
        a.set_xlabel("step")
        a.grid(True, alpha=0.3)
    fig.suptitle(f"{WANDB_NAME} · {GRPO_MODEL}", fontsize=11)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    if run.state in {"finished", "failed", "crashed"} or done_n >= target_steps:
        pbar.n = min(done_n, target_steps)
        pbar.refresh()
        pbar.close()
        print("done:", run.state, "steps=", done_n)
        break
    time.sleep(12)

## 4. Remote log tail (optional)

In [ ]:
print(
    subprocess.check_output(
        [
            "brev",
            "exec",
            INSTANCE,
            "tail -n 60 ~/teachonce/logs/grpo_dag_wandb.log 2>/dev/null || tail -n 60 ~/teachonce/logs/grpo_dag_launch.out",
        ],
        text=True,
    )
)